In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

def preprocess(train_file, test_file):
    train = np.load(train_file, allow_pickle=True)
    test = np.load(test_file, allow_pickle=True)

    X_train, y_train = train["X"], train["y"]
    X_test, y_test = test["X"], test["y"]

    ...

    return (X_train, y_train), (X_test, y_test)


(X_train, y_train), (X_test, y_test) = preprocess("train2.npz", "test2.npz")

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        ...

    def forward(self, x):
        ...


model = Model()

In [ ]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct = 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y)
        correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


def test(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * len(y)
            correct += (outputs.argmax(dim=1) == y).sum().item()

    return total_loss / len(loader.dataset), correct / len(loader.dataset)


train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)
test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 15

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = test(model, test_loader, criterion, device)

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2%} | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2%}"
    )

torch.save(model.state_dict(), "model2.pth")